# 04 — Configured Model Training and Persistence

**Objective:** Load models from YAML, compare all enabled candidates on validation, test only the winner, and persist lineage.

In [1]:
from pathlib import Path
import os, sys
import pandas as pd
import plotly.express as px

_cwd = Path.cwd().resolve()
ROOT = _cwd.parent if _cwd.name == "dev" else _cwd
sys.path.insert(0, str(ROOT / "src"))
os.environ.setdefault("MPLCONFIGDIR", "/tmp/credit-risk-lab-matplotlib")

from credit_risk_lab.config.settings import settings
print(f"Project root: {settings.project_root}")

Project root: /Users/surelmanda/Mlops-Databricks-Projects/credit-risk-lab


## 1. Inspect `configs/models.yaml` and instantiate candidates

In [2]:
from credit_risk_lab.infrastructure.modeling import build_configured_models, load_models_config

models_config = load_models_config()
models = build_configured_models(random_state=settings.random_state, config=models_config)
pd.DataFrame([{"model": m.name, "parameters": m.parameters, "early_stopping": m.early_stopping_rounds} for m in models])

,model,parameters,early_stopping
0,LogisticRegression,"{'max_iter': 2000, 'solver': 'lbfgs'}",NaN
1,RandomForest,"{'n_estimators': 300, 'max_depth': 12, 'min_sa...",NaN
2,XGBoost,"{'n_estimators': 120, 'learning_rate': 0.06, '...",15.0
3,CatBoost,"{'iterations': 120, 'learning_rate': 0.06, 'de...",15.0
4,LightGBM,"{'n_estimators': 120, 'learning_rate': 0.06, '...",15.0


## 2. Load and engineer only the 90% training partition

In [3]:
from credit_risk_lab.infrastructure.data_sources import CSVDataSourceConfig, CSVDatasetRepository
from credit_risk_lab.infrastructure.feature_engineering import LoanFeatureEngineer

train_raw = CSVDatasetRepository(CSVDataSourceConfig(path=settings.train_path)).load()
train_features = LoanFeatureEngineer().transform(train_raw)

2026-07-11 09:54:58 | INFO     | CSVDatasetRepository | credit_risk_lab.infrastructure.data_sources.csv_dataset_repository:load:51 - Chargement du fichier : /Users/surelmanda/Mlops-Databricks-Projects/credit-risk-lab/data/processed/train.csv
2026-07-11 09:54:58 | INFO     | CSVDatasetRepository | credit_risk_lab.infrastructure.data_sources.csv_dataset_repository:load:59 - Dataset chargé (40493 lignes, 14 colonnes)
2026-07-11 09:54:58 | INFO     | loan_feature_engineer | credit_risk_lab.infrastructure.feature_engineering.pandas_feature_engineer:_log_shape:126 - [FeatureEngineering] Entrée - shape = (40493, 14)
2026-07-11 09:54:58 | INFO     | loan_feature_engineer | credit_risk_lab.infrastructure.feature_engineering.pandas_feature_engineer:_log_shape:126 - [FeatureEngineering] Sortie - shape = (40493, 39)


## 3. Train, select on validation, and test the locked winner

In [4]:
from credit_risk_lab.application import TrainBoostingModelsUseCase

result = TrainBoostingModelsUseCase().execute(train_features)
display(result.metrics.round(4).rename_axis("validation candidates"))
display(result.test_metrics.round(4).rename_axis("internal final test of winner"))

2026-07-11 09:54:58 | WARNING  | TrainBoostingModelsUseCase | credit_risk_lab.application.train_boosting_models:execute:64 - Using random split: results are educational, not production evidence
2026-07-11 09:54:58 | INFO     | TrainBoostingModelsUseCase | credit_risk_lab.application.train_boosting_models:execute:75 - Dataset split completed: [{'split': 'train', 'rows': 24295, 'positive_rate': 0.22226795636962338}, {'split': 'validation', 'rows': 8099, 'positive_rate': 0.22224966045190764}, {'split': 'test', 'rows': 8099, 'positive_rate': 0.22224966045190764}]
2026-07-11 09:54:58 | INFO     | TrainBoostingModelsUseCase | credit_risk_lab.application.train_boosting_models:execute:80 - Excluding sensitive columns from training features: ['person_gender', 'is_female']
2026-07-11 09:54:59 | INFO     | TrainBoostingModelsUseCase | credit_risk_lab.application.train_boosting_models:execute:105 - Training LogisticRegression
2026-07-11 09:54:59 | INFO     | TrainBoostingModelsUseCase | credit_ris

,model,threshold,accuracy,precision,recall,f1,roc_auc,log_loss,brier,mcc,cohen_kappa,ece,pr_auc,ks,gini,balanced_accuracy
validation candidates,,,,,,,,,,,,,,,,
0,LightGBM,0.2484,0.9052,0.7289,0.9128,0.8106,0.9759,0.1563,0.0495,0.7568,0.7484,0.0084,0.9335,0.8158,0.9518,0.9079
1,XGBoost,0.2456,0.8981,0.7126,0.9078,0.7984,0.9735,0.1642,0.0519,0.7412,0.7316,0.0114,0.9275,0.8032,0.9470,0.9016
2,CatBoost,0.2391,0.8931,0.6992,0.9106,0.7910,0.9722,0.1681,0.0532,0.7322,0.7208,0.0101,0.9245,0.7986,0.9445,0.8993
3,RandomForest,0.4892,0.9001,0.7211,0.8978,0.7998,0.9709,0.2035,0.0666,0.7421,0.7343,0.0646,0.9202,0.7986,0.9418,0.8993
4,LogisticRegression,0.1806,0.8550,0.6144,0.9339,0.7412,0.9574,0.2148,0.0695,0.6737,0.6464,0.0153,0.8654,0.7664,0.9147,0.8832


,model,threshold,accuracy,precision,recall,f1,roc_auc,log_loss,brier,mcc,cohen_kappa,ece,pr_auc,ks,gini,balanced_accuracy
internal final test of winner,,,,,,,,,,,,,,,,
0,LightGBM,0.2484,0.9012,0.722,0.9033,0.8026,0.9736,0.1643,0.052,0.746,0.7378,0.0087,0.9271,0.8078,0.9471,0.902


## 4. Learning curves and validation comparison

In [5]:
from credit_risk_lab.infrastructure.visualization import plot_learning_curves, plot_model_comparison

plot_learning_curves(result.histories).show()
plot_model_comparison(result.metrics).show()

## 5. Persist the selected bundle with lineage

In [6]:
import subprocess
from credit_risk_lab.infrastructure.modeling import save_model_bundle, sha256_file

winner = result.selected_model_name
winner_row = result.test_metrics.iloc[0]
try:
    commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
except Exception:
    commit = "unavailable"
save_model_bundle(
    settings.model_bundle_path,
    model=result.models[winner], preprocessor=result.preprocessor,
    threshold=float(winner_row["threshold"]),
    metadata={
        "model_name": winner,
        "test_metrics": winner_row.to_dict(),
        "selection_metric": settings.selection_metric,
        "split_strategy": result.split_strategy,
        "training_dataset_sha256": sha256_file(settings.train_path),
        "external_test_dataset_sha256": sha256_file(settings.test_path),
        "models_config_sha256": sha256_file(settings.models_config_path),
        "git_commit": commit,
        "target_definition": "loan_status=1 is the synthetic positive risk class",
    },
)
settings.model_bundle_path

PosixPath('/Users/surelmanda/Mlops-Databricks-Projects/credit-risk-lab/models/best_boosting_model.joblib')